# OcuWeave · data quality and development errors
Marchel Andrian Shevchenko · Data Sorcerers · 2026 · software v0.3.0

This CPU-compatible notebook audits acquired FairVLMed arrays and original development notes, then diagnoses the frozen v0.2.0 OTFID checkpoints. It does not retrain models or tune on test performance. Numeric checks do not establish clinical validity. Dataset records, notes and images stay in private Drive; no raw note text is printed.

## Context and methods
Use the existing `OcuWeave — Datasets and experiments` folder. FairVLMed remains a separate glaucoma cohort. Original dataset files, official splits and pilot test results are preserved. Patient independence is source-reported for FairVLMed and not independently verified here. See `docs/protocols/quality_review.md` for the predeclared diagnostic bins and lexical patterns.


## 1. Connect Drive and pin the code

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, os, json, hashlib, shutil
from IPython.display import display, Image, Markdown
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/OcuWeave — Datasets and experiments')
assert PROJECT.is_dir(), 'Existing research folder required'
RELEASE = 'v0.3.0'
REPO = Path('/content/ocuweave-quality-' + RELEASE)
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',RELEASE,'https://github.com/chenko-the-sorcerers/ocuweave.git',str(REPO)],check=True,capture_output=True)
assert subprocess.check_output(['git','describe','--tags','--exact-match'],cwd=REPO,text=True).strip() == RELEASE
assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=REPO,text=True).strip()
REVISION = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
OUTPUT = PROJECT/'02 — Audits'/'quality-review-v0.3.0'
OUTPUT.mkdir(exist_ok=True)
LOGS = OUTPUT/'logs'
LOGS.mkdir(exist_ok=True)
def run(args, name):
    result = subprocess.run(args,cwd=REPO,capture_output=True,text=True)
    (LOGS/name).write_text(result.stdout+result.stderr)
    print(name, 'exit:', result.returncode, (result.stdout+result.stderr)[-1800:])
    result.check_returncode()
run([sys.executable,'-m','pip','install','--no-deps','-e','.'],'install.log')
import torch, numpy, matplotlib
print({'revision':REVISION,'torch':str(torch.__version__),'device':'cpu'})


## 2. Validate implementation
Synthetic tests cover corrupt arrays, object rejection, mismatched labels, duplicate pixels, privacy, split guards and end-to-end checkpoint diagnostics.

In [ ]:
run(['cmake','-S','.','-B','build','-DBUILD_TESTING=ON'],'cmake.log')
run(['cmake','--build','build'],'build.log')
run(['ctest','--test-dir','build','--output-on-failure'],'ctest.log')
os.environ['OCUWEAVE_NATIVE_LIBRARY'] = str(REPO/'build/libocuweave_native.so')
run([sys.executable,'-m','unittest','discover','-s','tests','-v'],'tests.log')
run([sys.executable,'scripts/audit_data.py','examples/synthetic_manifest.json'],'synthetic-audit.log')


## 3. Prepare verified runtime copies
Copy the two archives into disposable Colab storage for faster reads. Verify each copy against the previous acquisition receipt. No download or edit of the original dataset is needed.

In [ ]:
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()
def verified_copy(source,target,expected):
    target.parent.mkdir(parents=True,exist_ok=True)
    if not target.exists():
        assert shutil.disk_usage(target.parent).free > source.stat().st_size + 512*1024**2
        shutil.copyfile(source,target)
    assert digest(target)==expected, 'Checksum mismatch; preserve and inspect'
FAIR_SOURCE = PROJECT/'01 — Raw datasets/FairVLMed'
FAIR = Path('/content/fairvlmed-quality-input')
receipt=json.loads((FAIR_SOURCE/'download_manifest.json').read_text())
for item in receipt['files']:
    verified_copy(FAIR_SOURCE/item['path'], FAIR/item['path'], item['sha256'])
shutil.copyfile(FAIR_SOURCE/'download_manifest.json',FAIR/'download_manifest.json')
OTFID_SOURCE=PROJECT/'01 — Raw datasets/OTFID_v3'
ARCHIVE=Path('/content/ocuweave-quality-otfid.zip')
verified_copy(OTFID_SOURCE/'Ocular_Toxoplasmosis_Data_V3.zip',ARCHIVE,json.loads((OTFID_SOURCE/'download_manifest.json').read_text())['sha256'])
MANIFEST=PROJECT/'02 — Audits/otfid-v3-binary-v2-seed42-v0.2.0/manifest.json'
PILOT=PROJECT/'04 — Runs/v0.2.0/pilot-seed42-20epochs'
TEST_HASHES={p.name:digest(p) for p in PILOT.glob('*-test.json')}
assert len(TEST_HASHES)==2
print('Two archives and FairVLMed CSV verified; existing test-result hashes recorded.')


## 4. Audit numeric images and development-note review flags
All official splits receive technical image integrity checks. Only training/validation original notes receive lexical screening. Generated summaries and test note contents are excluded. Review flags do not resolve negation, temporality, laterality or clinical meaning.

In [ ]:
FAIR_OUT=OUTPUT/'fairvlmed'
if not FAIR_OUT.exists():
    run([sys.executable,'scripts/audit_fairvlmed_images.py','--root',str(FAIR),'--output',str(FAIR_OUT)],'fairvlmed-audit.log')
else:
    assert (FAIR_OUT/'summary.json').exists(), 'Incomplete audit: preserve and choose a new output'
fair=json.loads((FAIR_OUT/'summary.json').read_text())
assert fair['archive_sha256']==digest(FAIR/'Dataset/dataset.zip')
assert fair['csv_sha256']==digest(FAIR/'ReadMe/data_summary.csv')
notes=json.loads((FAIR_OUT/'note_review.json').read_text())
print(json.dumps(fair,indent=2))
print('Development-note review counts:',json.dumps(notes['counts'],indent=2))
print('Semantic leakage review completed:',notes['semantic_review_complete'])


## 5. Diagnose frozen OTFID checkpoints
Training/validation only; fixed probability threshold 0.5 and original model input size. Validation already selected checkpoints, so these are descriptive diagnostics, not independent performance estimates.

In [ ]:
diagnostics={}
for architecture in ('query','fcn'):
    checkpoint=PILOT/architecture/'best.pt'
    assert digest(checkpoint)==json.loads((PILOT/architecture/'checkpoint.json').read_text())['sha256']
    for split in ('train','validation'):
        target=OUTPUT/f'otfid-{architecture}-{split}'
        if not target.exists():
            run([sys.executable,'scripts/diagnose_segmentation.py','--manifest',str(MANIFEST),'--archive',str(ARCHIVE),'--checkpoint',str(checkpoint),'--output',str(target),'--split',split,'--device','cpu','--allow-unverified-patients'],f'{architecture}-{split}.log')
        report=json.loads((target/'diagnostics.json').read_text())
        assert report['checkpoint_sha256']==digest(checkpoint) and report['manifest_sha256']==digest(MANIFEST)
        diagnostics[f'{architecture}-{split}']=report
assert TEST_HASHES=={p.name:digest(p) for p in PILOT.glob('*-test.json')}
provenance={'software':RELEASE,'source_commit':REVISION,'fairvlmed_archive_sha256':fair['archive_sha256'],'otfid_manifest_sha256':digest(MANIFEST),'preserved_test_sha256':TEST_HASHES,'new_training':False,'device':'cpu'}
(OUTPUT/'provenance.json').write_text(json.dumps(provenance,indent=2))
print('Diagnostics saved. Both original test evaluations are unchanged.')


## Results
Numbers below are calculated from the saved reports. A technical audit pass is not clinical approval.

In [ ]:
lines=['| Model / split | Positive images | Positive-image Dice | Healthy images | Any-pixel false-positive rate |','|---|---:|---:|---:|---:|']
for key,report in diagnostics.items():
    m=report['metrics']
    lines.append(f"| {key} | {m['positive_samples']} | {m['positive_macro_dice']:.4f} | {m['empty_samples']} | {m['empty_image_false_positive_rate']:.1%} |")
summary=f"FairVLMed: {fair['decoded']}/{fair['rows']} numeric images decoded; {fair['failed']} failures; {fair['label_mismatches']} label mismatches; {fair['cross_split_exact_pixel_groups']} exact-pixel groups cross official splits.\n\n"+'\n'.join(lines)
summary+='\n\nNo new training. Test evaluations are preserved. Note flags require semantic review; patient independence and clinical image quality remain unverified.'
(OUTPUT/'README.md').write_text('# OcuWeave quality review v0.3.0\n\n'+summary)
display(Markdown(summary))


## Diagnostic figures
The area-strata plot describes validation errors. The failure panel is intentionally selected for worst positive Dice and largest healthy false-positive area; it is not a representative performance illustration.

In [ ]:
display(Image(filename=str(OUTPUT/'otfid-query-validation/strata.png'),width=900))
display(Image(filename=str(OUTPUT/'otfid-query-validation/review_panel.png'),width=750))


## Takeaways and next decisions
Use observed strata and selected failures to propose changes using development data only. Verify annotation quality and patient grouping before stronger claims. Review FairVLMed temporal/diagnostic leakage before training a clinical-text branch. Do not infer early disease origin or future visual loss from this pilot. No records are automatically deleted or relabeled.

References: [FairVLMed official dataset card](https://huggingface.co/datasets/harvardairobotics/FairVLMed); Y. Luo et al., “FairCLIP: Harnessing Fairness in Vision-Language Learning,” CVPR 2024, pp. 12289–12301; [OTFID v3](https://zenodo.org/records/13883485). Dataset licenses remain separate from the code license.
